In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

print("=========================================")
print("  E-COMMERCE FEATURE ENGINEERING")
print("=========================================\n")

# 1. Load the raw data
df = pd.read_csv('../../data/raw/online_shoppers_intention.csv')

# 2. Map Target and Boolean Variables to Int
df['Revenue'] = df['Revenue'].astype(int)
df['Weekend'] = df['Weekend'].astype(int)

# 3. Apply Log Transformation to Skewed Features
skewed_cols = ['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration', 'PageValues']
df_transformed = df.copy()
for col in skewed_cols:
    df_transformed[col] = np.log1p(df_transformed[col])

# 4. One-Hot Encode Categorical Variables
categorical_cols = ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType']
df_encoded = pd.get_dummies(df_transformed, columns=categorical_cols, drop_first=True)

# 5. Separate Features (X) and Target (y)
X = df_encoded.drop("Revenue", axis=1)
y = df_encoded["Revenue"]

# Save FULL processed dataset before splitting/scaling (for future experiments)
os.makedirs('../../data/processed', exist_ok=True)
full_df = pd.concat([X, y], axis=1)
full_df.to_csv('../../data/processed/clean_dataset.csv', index=False)
print("Saved full engineered dataset (unscaled) to clean_dataset.csv\n")

# 6. Stratified Train-Test Split (CRITICAL: Do this BEFORE scaling)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("--- Train Class Distribution ---")
print(y_train.value_counts(normalize=True))
print("\n--- Test Class Distribution ---")
print(y_test.value_counts(normalize=True))
print("\n(Validation: Stratification successfully maintained the 84/16 split)\n")

# 7. Scale Numerical Features (Prevent Data Leakage)
scaler = StandardScaler()
cols_to_scale = [
    'Administrative', 'Administrative_Duration', 'Informational', 
    'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]

# FIT on Train only, TRANSFORM on Train and Test
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# 8. Save Production Artifacts (Unified Preprocessor)
os.makedirs('../../backend/models', exist_ok=True)

artifacts = {
    "scaler": scaler,
    "feature_columns": X.columns.tolist()
}

with open('../../backend/models/preprocessor.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print("--- Artifacts Saved ---")
print("Saved preprocessor.pkl (Contains Scaler and Exact Feature Column list for FastAPI)\n")

# 9. Save Processed, Scaled Splits for Model Training
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv('../../data/processed/train.csv', index=False)
test_df.to_csv('../../data/processed/test.csv', index=False)

print("--- Data Saved to /data/processed/ ---")
print(f"Train set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print("\nFeature Engineering Pipeline Complete!")